The Advanced Computing Center for Research and Education
Project Overview The Advanced Computing Center for Research and Education (ACCRE) operates Vanderbilt University's high-performance computing cluster. Jobs submitted to ACCRE are managed by the slurm scheduler, which tracks compute and memory resources.

ACCRE staff have hypothesized that the scheduler sometimes becomes unresponsive because it is processing large bursts of job completions. This especially affects automated job submitters, such as members of the Open Science Grid.

Your goal is to evaluate whether the data supports the hypothesis of bursts of job completions contributing to scheduler unresponsiveness.

You are provided three datasets:

fullsample.csv: Contains slurm job records. Job completions correspond to jobs in the "COMPLETED" state with exit code "0:0".
slurm_wrapper_ce5.log, slurm_wrapper_ce6.log: These log files contain every slurm command executed by the CE5 and CE6 servers (gateways to the Open Science Grid).
Unresponsive periods are indicated by "sbatch" commands from user 9204 that have:
return code = 1
execution time > 15 seconds

In [1]:
import pandas as pd

**Phase 1: Explore the Data**

Objectives:

Understand the purpose of each dataset.
Inspect column types, sizes, and example rows.

In [2]:
jobs = pd.read_csv("data/fullsample.csv")
jobs.head(5)

,JOBID,STATE,BEGIN,END,REQMEM,USEDMEM,REQTIME,USEDTIME,NODES,CPUS,PARTITION,EXITCODE
0,30616928,RUNNING,2021-07-31T22:15:00,Unknown,2048Mn,0,10:04:00,67-22:14:22,1,1,production,0:0
1,30853133,COMPLETED,2021-08-06T11:36:09,2021-09-05T11:36:32,262144Mn,20604.62M,30-00:00:00,30-00:00:23,1,1,cgw-platypus,0:0
2,30858137,COMPLETED,2021-08-06T19:04:39,2021-09-05T19:04:53,204800Mn,57553.77M,30-00:00:00,30-00:00:14,1,32,cgw-tbi01,0:0
3,30935078,COMPLETED,2021-08-09T16:52:51,2021-09-07T20:52:55,65536Mn,20577.96M,29-04:00:00,29-04:00:04,1,8,cgw-platypus,0:0
4,31364111_2,COMPLETED,2021-08-17T07:45:07,2021-09-10T16:45:24,16384Mn,9733.43M,24-09:00:00,24-09:00:17,1,1,production,0:0


In [3]:
jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7395885 entries, 0 to 7395884
Data columns (total 12 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   JOBID      object
 1   STATE      object
 2   BEGIN      object
 3   END        object
 4   REQMEM     object
 5   USEDMEM    object
 6   REQTIME    object
 7   USEDTIME   object
 8   NODES      int64 
 9   CPUS       int64 
 10  PARTITION  object
 11  EXITCODE   object
dtypes: int64(2), object(10)
memory usage: 677.1+ MB


In [4]:
print(jobs.columns)

Index(['JOBID', 'STATE', 'BEGIN', 'END', 'REQMEM', 'USEDMEM', 'REQTIME',
       'USEDTIME', 'NODES', 'CPUS', 'PARTITION', 'EXITCODE'],
      dtype='object')


In [5]:
total_nan = jobs.isna().sum()
print(total_nan)

JOBID        0
STATE        0
BEGIN        0
END          0
REQMEM       0
USEDMEM      0
REQTIME      0
USEDTIME     0
NODES        0
CPUS         0
PARTITION    0
EXITCODE     0
dtype: int64


In [6]:
ce5 = pd.read_csv("data/slurm_wrapper_ce5.log",
                  header=None,
                  delimiter=' - ',
                  engine='python')

ce5.rename(columns={
    0: "Date",
    1: "User",
    2: "Retry",
    3: "Time",
    4: "Returncode",
    5: "Command"
}, inplace=True)


In [7]:
ce5.shape

(4770893, 6)

In [8]:
ce5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4770893 entries, 0 to 4770892
Data columns (total 6 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   Date        object
 1   User        object
 2   Retry       object
 3   Time        object
 4   Returncode  object
 5   Command     object
dtypes: object(6)
memory usage: 218.4+ MB


* The ce5 data contains about 4.77M rows and 6 columns

In [9]:
ce5.head()

,Date,User,Retry,Time,Returncode,Command
0,2020-10-16 08:15:39.278699,user 0,retry 0,time 0.07347559928894043,returncode 0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
1,2020-10-16 08:18:08.313309,user 0,retry 0,time 0.18363237380981445,returncode 0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
2,2020-10-16 08:22:48.128689,user 0,retry 0,time 0.07547116279602051,returncode 0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
3,2020-10-16 08:25:13.257408,user 0,retry 0,time 0.09484362602233887,returncode 0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
4,2020-10-16 08:31:01.460723,user 0,retry 0,time 0.07498788833618164,returncode 0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."


In [10]:
ce6 = pd.read_csv("data/slurm_wrapper_ce6.log",
                  header=None,
                  delimiter=' - ',
                  engine='python')

ce6.columns = ["Date", "User", "Retry", "Time", "Returncode", "Command"]
ce6.rename(columns={
    0: "Date",
    1: "User",
    2: "Retry",
    3: "Time",
    4: "Returncode",
    5: "Command"
}, inplace=True)


In [11]:
ce6.head(5)

,Date,User,Retry,Time,Returncode,Command
0,2020-10-16 10:37:44.163454,user 9202,retry 0,time 0.08495402336120605,returncode 0,"command ['/usr/bin/scontrol', 'show', 'job', '..."
1,2020-10-16 10:37:44.206654,user 9202,retry 0,time 0.08943057060241699,returncode 0,"command ['/usr/bin/scontrol', 'show', 'job', '..."
2,2020-10-16 10:37:44.218760,user 9202,retry 0,time 0.05928945541381836,returncode 0,"command ['/usr/bin/scontrol', 'show', 'job', '..."
3,2020-10-16 10:37:44.256403,user 9202,retry 0,time 0.038695573806762695,returncode 0,"command ['/usr/bin/scontrol', 'show', 'job', '..."
4,2020-10-16 10:37:44.611603,user 9202,retry 0,time 0.03343677520751953,returncode 0,"command ['/usr/bin/scontrol', 'show', 'job', '..."


* The column names were changed from numerical to text format.
* Both the ce5 and ce6 data type contained mixed data types, such as numbers and text.

In [12]:
combined_ce5_ce6 = pd.concat([ce5, ce6], ignore_index=True)


* Combined both cs5 and ce6 DataFrames.

In [14]:
combined_ce5_ce6['User'] = (
    combined_ce5_ce6['User']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype("Int64")
)

combined_ce5_ce6['Retry'] = (
    combined_ce5_ce6['Retry']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype("Int64")
)

combined_ce5_ce6['Returncode'] = (
    combined_ce5_ce6['Returncode']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype("Int64")
)

combined_ce5_ce6['Time'] = (
    combined_ce5_ce6['Time']
    .astype(str)
    .str.extract(r'(\d+\.\d+)')
    .astype(float)
)

In [15]:
combined_ce5_ce6.head(5)

,Date,User,Retry,Time,Returncode,Command
0,2020-10-16 08:15:39.278699,0,0,0.073476,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
1,2020-10-16 08:18:08.313309,0,0,0.183632,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
2,2020-10-16 08:22:48.128689,0,0,0.075471,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
3,2020-10-16 08:25:13.257408,0,0,0.094844,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."
4,2020-10-16 08:31:01.460723,0,0,0.074988,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '..."


* The ce5 and ce6 DataFrames originally contained mixed data types, including numbers and text. These DataFrames were then combined and transformed into clean DataFrames.

**Phase 2: Clean and Transform the Data**

Objectives:

* Extract job completions from fullsample.csv.

* Parse CE5 and CE6 logs to identify unresponsive events.

Rule:
Unresponsive periods are indicated by "sbatch" commands from user 9204 that have:
return code = 1
execution time > 15 seconds

* Parse and clean the columns in the ce5 and ce6 DataFrames.
* Identify unresponsive events in both DataFrames.

In [16]:
unresponsive = combined_ce5_ce6[
    (combined_ce5_ce6['User'] == 9204) &
    (combined_ce5_ce6['Returncode'] == 1) &
    (combined_ce5_ce6['Time'] > 15) &
    (combined_ce5_ce6['Command'].str.contains("sbatch", case=False, na=False))
]

print("Total unresponsive events:", len(unresponsive))
print(unresponsive.head())

Total unresponsive events: 3296
                             Date  User  Retry       Time  Returncode  \
49958  2020-10-18 06:53:44.272915  9204      0  20.038464           1   
49972  2020-10-18 06:54:04.322412  9204      1  20.048906           1   
50467  2020-10-18 07:47:25.825172  9204      0  20.082628           1   
50473  2020-10-18 07:47:45.871008  9204      1  20.045221           1   
50582  2020-10-18 07:53:33.972840  9204      0  20.041486           1   

                                                 Command  
49958  command ['/usr/bin/sbatch', '/tmp/condor_g_scr...  
49972  command ['/usr/bin/sbatch', '/tmp/condor_g_scr...  
50467  command ['/usr/bin/sbatch', '/tmp/condor_g_scr...  
50473  command ['/usr/bin/sbatch', '/tmp/condor_g_scr...  
50582  command ['/usr/bin/sbatch', '/tmp/condor_g_scr...  


* The number of unresponsive events was identified in the both ce5 and ce6 datasets.

* Extract job completions from fullsample.csv.

In [35]:
jobs_completed = jobs[jobs['STATE'].str.contains('COMPLETED', na=False)].copy()
jobs_completed

,JOBID,STATE,BEGIN,END,REQMEM,USEDMEM,REQTIME,USEDTIME,NODES,CPUS,PARTITION,EXITCODE
1,30853133,COMPLETED,2021-08-06T11:36:09,2021-09-05T11:36:32,262144Mn,20604.62M,30-00:00:00,30-00:00:23,1,1,cgw-platypus,0:0
2,30858137,COMPLETED,2021-08-06T19:04:39,2021-09-05T19:04:53,204800Mn,57553.77M,30-00:00:00,30-00:00:14,1,32,cgw-tbi01,0:0
3,30935078,COMPLETED,2021-08-09T16:52:51,2021-09-07T20:52:55,65536Mn,20577.96M,29-04:00:00,29-04:00:04,1,8,cgw-platypus,0:0
4,31364111_2,COMPLETED,2021-08-17T07:45:07,2021-09-10T16:45:24,16384Mn,9733.43M,24-09:00:00,24-09:00:17,1,1,production,0:0
5,31364111_3,COMPLETED,2021-08-17T07:45:07,2021-09-06T16:17:34,16384Mn,9708.04M,24-09:00:00,20-08:32:27,1,1,production,0:0
...,...,...,...,...,...,...,...,...,...,...,...,...
7395880,25493434,COMPLETED,2020-10-31T23:39:00,2020-10-31T23:40:46,2000Mn,0.09M,2-00:00:00,00:01:46,1,1,sam,0:0
7395881,25493435,COMPLETED,2020-10-31T23:39:13,2020-10-31T23:40:38,2000Mn,187.92M,2-00:00:00,00:01:25,1,1,sam,0:0
7395882,25493476,COMPLETED,2020-10-31T23:46:29,2020-10-31T23:49:43,4096Mc,803.97M,12:00:00,00:03:14,1,1,production,0:0
7395883,25493515,COMPLETED,2020-10-31T23:49:44,2020-10-31T23:51:40,2000Mn,0.09M,2-00:00:00,00:01:56,1,1,sam,0:0


* Converted the 'STATE' column to string type to ensure consistent text operations and avoid errors caused by missing or non-string values.
* Filtered the dataset to include only jobs whose 'STATE' contains the substring 'COMPLETED' (case-insensitive).
* Created a separate DataFrame 'jobs_completed' to isolate completed jobs for focused analysis.

**Create analysis-ready features (time windows, completion counts, unresponsiveness indicators).**

In [54]:
combined_data  = pd.concat([combined_ce5_ce6, jobs],ignore_index=True, sort=False)
combined_data

,Date,User,Retry,Time,Returncode,Command,JOBID,STATE,BEGIN,END,REQMEM,USEDMEM,REQTIME,USEDTIME,NODES,CPUS,PARTITION,EXITCODE
0,2020-10-16 08:15:39.278699,0,0,0.073476,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-10-16 08:18:08.313309,0,0,0.183632,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-10-16 08:22:48.128689,0,0,0.075471,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-10-16 08:25:13.257408,0,0,0.094844,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-10-16 08:31:01.460723,0,0,0.074988,0,"command ['/usr/bin/sacct', '-u', 'appelte1', '...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16943293,NaN,<NA>,<NA>,NaN,<NA>,NaN,25493434,COMPLETED,2020-10-31T23:39:00,2020-10-31T23:40:46,2000Mn,0.09M,2-00:00:00,00:01:46,1.0,1.0,sam,0:0
16943294,NaN,<NA>,<NA>,NaN,<NA>,NaN,25493435,COMPLETED,2020-10-31T23:39:13,2020-10-31T23:40:38,2000Mn,187.92M,2-00:00:00,00:01:25,1.0,1.0,sam,0:0
16943295,NaN,<NA>,<NA>,NaN,<NA>,NaN,25493476,COMPLETED,2020-10-31T23:46:29,2020-10-31T23:49:43,4096Mc,803.97M,12:00:00,00:03:14,1.0,1.0,production,0:0
16943296,NaN,<NA>,<NA>,NaN,<NA>,NaN,25493515,COMPLETED,2020-10-31T23:49:44,2020-10-31T23:51:40,2000Mn,0.09M,2-00:00:00,00:01:56,1.0,1.0,sam,0:0


* Combined three datasets into a single DataFrame suitable for analysis.

In [18]:
combined_data["Date"] = pd.to_datetime(combined_data["Date"], errors="coerce")

* Converted the 'Date' column to datetime objects.

In [52]:

# Create timestamp
combined_data['Timestamp'] = (
    combined_data['Date'].fillna(combined_data['BEGIN'])
)
combined_data['Timestamp'] = pd.to_datetime(combined_data['Timestamp'], errors='coerce')

# ❗ Drop rows with missing timestamps
combined_data = combined_data.set_index('Timestamp').sort_index()

# Rolling window counts
for window in ['1min', '5min', '15min']:
    combined_data[f'Events_{window}'] = combined_data['User'].rolling(window=window).count()



In [49]:
combined_data.head(5)

,Date,User,Retry,Time,Returncode,Command,JOBID,STATE,BEGIN,END,...,Completed_5min,Completed_15min,BEGIN_dt,END_dt,Missing_END,Missing_BEGIN,Duration,Running_1min,Running_5min,Running_15min
Timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-01 00:03:08,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460560,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:17,...,1.0,1.0,2020-10-01 00:03:08,2020-10-01 00:23:17,False,False,1209.0,0.0,0.0,0.0
2020-10-01 00:03:08,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460565,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:15,...,2.0,2.0,2020-10-01 00:03:08,2020-10-01 00:23:15,False,False,1207.0,0.0,0.0,0.0
2020-10-01 00:03:08,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460559,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:14,...,3.0,3.0,2020-10-01 00:03:08,2020-10-01 00:23:14,False,False,1206.0,0.0,0.0,0.0
2020-10-01 00:03:08,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460558,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:14,...,4.0,4.0,2020-10-01 00:03:08,2020-10-01 00:23:14,False,False,1206.0,0.0,0.0,0.0
2020-10-01 00:03:08,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460557,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:13,...,5.0,5.0,2020-10-01 00:03:08,2020-10-01 00:23:13,False,False,1205.0,0.0,0.0,0.0


* Completion counts.

In [47]:
completion_counts = combined_data['STATE'] == 'COMPLETED'
running_counts = combined_data['STATE'] == 'RUNNING'

for window in ['1min', '5min', '15min']:
    
    combined_data[f'Completed_{window}'] = completion_counts.rolling(window=window).sum()
    combined_data[f'Running_{window}'] = running_counts.rolling(window=window).sum()
    
print(f"Total completed jobs: {completion_counts.sum()}")
print(f"Total running jobs: {running_counts.sum()}")

Total completed jobs: 7375084
Total running jobs: 208


* Unresponsive events.

In [41]:
unresponsive = combined_data[
    (combined_data['User'] == 9204) &
    (combined_data['Returncode'] == 1) &
    (combined_data['Time'] > 15) &
    (combined_data['Command'].str.contains("sbatch", case=False, na=False))
]

print("Total unresponsive events:", len(unresponsive))
unresponsive.head()

Total unresponsive events: 3296


,Date,User,Retry,Time,Returncode,Command,JOBID,STATE,BEGIN,END,...,NODES,CPUS,PARTITION,EXITCODE,Events_1min,Events_5min,Events_15min,Completed_1min,Completed_5min,Completed_15min
Timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-18 06:16:25.392946,2020-10-18 06:16:25.392946,9204,0,20.037672,1,"command ['/usr/bin/sbatch', '/tmp/condor_g_scr...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,53.0,200.0,359.0,0.0,9.0,56.0
2020-10-18 06:38:44.172473,2020-10-18 06:38:44.172473,9204,0,20.038736,1,"command ['/usr/bin/sbatch', '/tmp/condor_g_scr...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,111.0,240.0,585.0,0.0,5.0,58.0
2020-10-18 06:53:44.272915,2020-10-18 06:53:44.272915,9204,0,20.038464,1,"command ['/usr/bin/sbatch', '/tmp/condor_g_scr...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,33.0,258.0,487.0,25.0,56.0,72.0
2020-10-18 06:54:04.322412,2020-10-18 06:54:04.322412,9204,1,20.048906,1,"command ['/usr/bin/sbatch', '/tmp/condor_g_scr...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,26.0,264.0,483.0,13.0,56.0,72.0
2020-10-18 07:47:25.825172,2020-10-18 07:47:25.825172,9204,0,20.082628,1,"command ['/usr/bin/sbatch', '/tmp/condor_g_scr...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,5.0,65.0,441.0,1.0,6.0,79.0


In [42]:
# Convert begin and end to datetime
combined_data['BEGIN_dt'] = pd.to_datetime(combined_data['BEGIN'], errors='coerce')
combined_data['END_dt'] = pd.to_datetime(combined_data['END'], errors='coerce')

# Unresponsive conditions
combined_data['Missing_END'] = combined_data['END_dt'].isna()
combined_data['Missing_BEGIN'] = combined_data['BEGIN_dt'].isna()

combined_data['Duration'] = (combined_data['END_dt'] - combined_data['BEGIN_dt']).dt.total_seconds()
combined_data

,Date,User,Retry,Time,Returncode,Command,JOBID,STATE,BEGIN,END,...,Events_5min,Events_15min,Completed_1min,Completed_5min,Completed_15min,BEGIN_dt,END_dt,Missing_END,Missing_BEGIN,Duration
Timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-01 00:03:08.000000,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460560,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:17,...,0.0,0.0,1.0,1.0,1.0,2020-10-01 00:03:08,2020-10-01 00:23:17,False,False,1209.0
2020-10-01 00:03:08.000000,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460565,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:15,...,0.0,0.0,2.0,2.0,2.0,2020-10-01 00:03:08,2020-10-01 00:23:15,False,False,1207.0
2020-10-01 00:03:08.000000,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460559,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:14,...,0.0,0.0,3.0,3.0,3.0,2020-10-01 00:03:08,2020-10-01 00:23:14,False,False,1206.0
2020-10-01 00:03:08.000000,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460558,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:14,...,0.0,0.0,4.0,4.0,4.0,2020-10-01 00:03:08,2020-10-01 00:23:14,False,False,1206.0
2020-10-01 00:03:08.000000,NaT,<NA>,<NA>,NaN,<NA>,NaN,24460557,COMPLETED,2020-10-01T00:03:08,2020-10-01T00:23:13,...,0.0,0.0,5.0,5.0,5.0,2020-10-01 00:03:08,2020-10-01 00:23:13,False,False,1205.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-10-07 21:59:35.014602,2021-10-07 21:59:35.014602,9221,0,0.060087,0,"command ['/usr/bin/squeue', '-o', '%i %T', '-u...",NaN,NaN,NaN,NaN,...,45.0,101.0,0.0,0.0,0.0,NaT,NaT,True,True,NaN
2021-10-07 21:59:35.238970,2021-10-07 21:59:35.238970,9202,0,0.098044,0,"command ['/usr/bin/squeue', '-o', '%i %T', '-u...",NaN,NaN,NaN,NaN,...,46.0,102.0,0.0,0.0,0.0,NaT,NaT,True,True,NaN
2021-10-07 21:59:57.265189,2021-10-07 21:59:57.265189,9203,0,0.024550,0,"command ['/usr/bin/squeue', '-o', '%i %T', '-u...",NaN,NaN,NaN,NaN,...,42.0,102.0,0.0,0.0,0.0,NaT,NaT,True,True,NaN
